# Période et profil replié

Le notebook lit un CSV déjà préparé, recherche une période puis trace le profil et la heatmap.

| Mode | Données | Méthode |
|---|---|---|
| `events` | Une ligne par événement | Epoch folding, χ² de Pearson |
| `values` | Une valeur par instant | Epoch folding, ANOVA L de Davies |

**CSV événements :**
```csv
timestamp
2025-12-01T00:00:00.123456789Z
2025-12-01T00:00:01.250000000Z
```

**CSV valeurs :**
```csv
timestamp,value
2025-12-01T00:00:00.123456789Z,125.5
2025-12-01T00:00:01.250000000Z,84.0
```

## 1. Paramètres

Renseigner `csv_path`, `mode`, la plage de périodes, `phase_bins`
et l’unité `value_label`.

Dépendances : pandas, NumPy, matplotlib. Exécuter les cellules dans l’ordre.


In [ ]:
from math import ceil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

csv_path = 'timeseries.csv'
mode = 'values' #values/events
p_min, p_max = (1.0, 40.0) #seconds
phase_bins = 50
value_label = 'Lifetime (ms)'
title = Path(csv_path).stem
plt.rcParams['figure.dpi'] = 120
cyan, gold, dark = ('#62e4df', '#f3cf86', '#080b17')

## 2. Lecture intégrale du CSV

Toutes les lignes du fichier sont analysées, sans filtre, tri ni nettoyage car la timeseries est supposée nettoyée.
Le premier timestamp définit l’origine ; le dernier définit la fin de la série.

En mode événements, cet intervalle est supposé observé sans interruption. Le fichier
ne permet pas de connaître les périodes silencieuses avant le premier ou après le dernier
événement : elles ne font donc pas partie de l’analyse. La borne numérique de fin est
placée juste après le dernier timestamp pour inclure aussi le dernier événement.


In [ ]:
data = pd.read_csv(csv_path)
dates = pd.to_datetime(data.timestamp, format = 'ISO8601', utc = True)
timestamps = pd.DatetimeIndex(dates).as_unit('ns').asi8
elapsed = timestamps-timestamps[0]
times = elapsed/1e9
values = data.value.to_numpy(float) if mode == 'values' else None

## 3. Recherche de période

Les données sont repliées sur chaque période candidate. Le code maximise :

- **Événements** : $\sum_j (n_j-e_j)^2/e_j$, où $e_j$ tient compte du temps
  d’exposition de chaque phase, y compris les cycles partiels aux bords.
- **Valeurs** : $L=[SSA/(K-1)]/[SSE/(N-K)]$, avec $K$ classes occupées,
  $N$ mesures, $SSA$ la dispersion entre classes et $SSE$ la dispersion résiduelle.

La grille est régulière en fréquence, avec un pas au plus égal à $1/(4MT)$
($M$ classes, durée $T$). Le calcul est effectué par lots pour limiter la mémoire.
Au moins cinq cycles sont requis ; la limite de deux millions d’essais impose
de réduire la plage de périodes si nécessaire.

La période retenue est une candidate : les alias et harmoniques restent possibles.
Aucune significativité n’est attribuée automatiquement.

Références : [Leahy et al., 1983](https://adsabs.harvard.edu/pdf/1983ApJ...272..256L)
et [Davies, 1990](https://adsabs.harvard.edu/pdf/1990MNRAS.244...93D).


In [ ]:
def phase_labels(times, periods, bins):
  """assign each observation to a phase bin"""
  phase = np.remainder(times, periods)/periods
  labels = (phase*bins).astype(int)
  
  return np.minimum(labels, bins-1)

def phase_exposure(periods, span, bins):
  """compute exposure for each phase over the full file"""
  periods = np.asarray(periods).reshape(-1, 1)
  cycles = np.floor(span/periods)
  remaining = span-cycles*periods
  phase_start = np.arange(bins)*(periods/bins)
  partial_exposure = np.clip(remaining-phase_start, 0, (periods/bins))
  
  return cycles*(periods/bins) + partial_exposure

def period_scores(times, values, periods, bins, span):
  """compute pearson or davies scores for a batch of periods"""
  labels = phase_labels(times, periods[:, None], bins)
  offsets = np.arange(len(periods))[:, None]*bins
  groups = labels + offsets
  
  flat_groups = groups.ravel()
  counts = np.bincount(flat_groups, minlength = len(periods)*bins)
  counts = counts.reshape(len(periods), bins)
  
  if values is None:
    exposure = phase_exposure(periods, span, bins)
    expected = (len(times)/span)*exposure #event_rate*exposure
    deviation = counts - expected
    contributions = deviation**2/expected
    return contributions.sum(axis = 1)
  
  centered = values-values.mean()
  repeated = np.tile(centered, len(periods))
  totals = np.bincount(flat_groups, weights = repeated,
    minlength = len(periods)*bins).reshape(len(periods), bins)
  
  means = totals/np.maximum(counts, 1)
  fitted = means.ravel()[groups]
  
  residual = centered-fitted
  
  between = np.sum(counts*means**2, axis = 1)
  within = np.sum(residual**2, axis = 1)
  occupied = (counts > 0).sum(axis = 1)
  between_df = occupied-1
  within_df = len(times)-occupied
  
  with np.errstate(divide = 'ignore', invalid = 'ignore'):
    between_variance = between/between_df
    within_variance = within/within_df
    scores = between_variance/within_variance
  invalid = (between_df <= 0) | (within_df <= 0)
  scores[invalid] = np.nan
  
  return scores

def find_period(times, values, p_min, p_max, bins):
  """select the best period on a uniform frequency grid"""
  span = np.nextafter(times[-1], np.inf)
  p_max = min(p_max, span/5)

  trials = ceil(((1/p_min) - (1/p_max))*(4*bins*span))+1 #ceil(freq_range*grid_density)+1
  frequencies = np.linspace(1/p_min, 1/p_max, trials)
  periods = 1/frequencies
  scores = np.full(trials, np.nan)
  batch = max(1, 262_144//len(times))
  
  for start in range(0, trials, batch):
    stop = min(start+batch, trials)
    candidates = periods[start:stop]
    scores[start:stop] = period_scores(times, values, candidates, bins, span)
    
  if np.isnan(scores).all(): raise ValueError('No identifiable period.')
  best = np.nanargmax(scores)
  return float(periods[best])

period_s = find_period(times, values, p_min, p_max, phase_bins)

## 4. Repliement

En mode événements, le profil représente un taux par seconde d’exposition.
En mode valeurs, il représente la moyenne des mesures par phase.
Les cases sans mesure restent manquantes ; les cases observées sans événement valent zéro.


In [ ]:
def fold_series(times, values, period, bins):
  """build the profile and cycle matrix with their weights"""
  span = np.nextafter(times[-1], np.inf)
  cycles = np.floor(times/period).astype(int)
  phases = phase_labels(times, period, bins)
  rows = cycles.max()+1
  cell_count = rows*bins
  
  if cell_count > 2_000_000: raise MemoryError('Reduce the heatmap size.')
  counts = np.bincount(cycles*bins+phases, minlength = cell_count).reshape(rows, bins)
  
  if values is None:
    totals = counts
    cycle_start = np.arange(rows)[:, None]*period
    phase_start = np.arange(bins)*period/bins
    weights = np.clip(span-cycle_start-phase_start, 0, period/bins)
  else:
    totals = np.bincount(cycles*bins+phases, weights = values, minlength = cell_count)
    totals = totals.reshape(rows, bins)
    weights = counts
  
  matrix = totals/np.where(weights > 0, weights, np.nan)
  phase_totals = totals.sum(axis = 0)
  phase_weights = weights.sum(axis = 0)
  profile = phase_totals/np.where(phase_weights > 0, phase_weights, np.nan)
  
  return profile, matrix, weights

profile, matrix, weights = fold_series(times, values, period_s, phase_bins)
feature = 'Events / second' if mode == 'events' else value_label

## 5. Graphique

La courbe montre le profil moyen sans lissage. La heatmap conserve les cycles manquants
en gris et utilise une échelle logarithmique de couleur, signée si nécessaire.
Pour l’affichage, les cycles contigus sont regroupés en au plus 250 lignes, pondérées
par l’exposition ou le nombre de mesures. La période apparaît dans le titre.


In [ ]:
rows, bins = matrix.shape
edges = np.linspace(0, rows, min(250, rows)+1, dtype = int)
totals = np.nan_to_num(matrix)*weights
totals = np.add.reduceat(totals, edges[:-1], axis = 0)
counts = np.add.reduceat(weights, edges[:-1], axis = 0)
means = totals/np.where(counts > 0, counts, np.nan)

colors = np.sign(means)*np.log1p(np.abs(means))
phases = np.linspace(0, 1, bins+1)
centers = (phases[:-1]+phases[1:])/2
cmap = plt.get_cmap('magma').with_extremes(bad = '#303342')

with plt.style.context('dark_background'):
  fig, (top, bottom) = plt.subplots(2, 1, figsize = (13, 7),
    sharex = True, height_ratios = [1, 3], constrained_layout = True)
  top.plot(centers, profile, color = cyan)
  top.set(ylabel = feature, title = f"{title} | P = {period_s:.6f} s")

  mesh = bottom.pcolormesh(phases, edges, colors, cmap = cmap)
  bottom.invert_yaxis()
  bottom.set(xlabel = 'Phase (cycles)', ylabel = 'Cycle number', xlim = (0, 1))
  fig.colorbar(mesh, ax = [top, bottom], label = f"{'signed log(1+|mean|)' if np.any(means < 0) else 'log(1+mean)'}: {feature}")
  plt.show()